In [ ]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 54.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving orders_cleaned.csv to orders_cleaned.csv


In [ ]:
df = pd.read_csv("orders_cleaned.csv")

 Basic check

In [ ]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,0000-00-00 00:00:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,0000-00-00 00:00:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,0000-00-00 00:00:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,0000-00-00 00:00:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,0000-00-00 00:00:00,3.39,17850,United Kingdom


Step 2: Data Validation
-- Data shape
-- Missing values
-- Duplicate rows

In [ ]:
df.shape

(2000, 8)

In [ ]:
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,6
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,0
Country,0


In [ ]:
df.duplicated().sum()

np.int64(36)

EDA: Basic Data Understanding

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    2000 non-null   object 
 1   StockCode    2000 non-null   object 
 2   Description  1994 non-null   object 
 3   Quantity     2000 non-null   int64  
 4   InvoiceDate  2000 non-null   object 
 5   UnitPrice    2000 non-null   float64
 6   CustomerID   2000 non-null   int64  
 7   Country      2000 non-null   object 
dtypes: float64(1), int64(2), object(5)
memory usage: 125.1+ KB


In [ ]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,2000.000000,2000.000000,2000.000000
mean,9.200000,3.799075,11456.583500
std,28.289418,13.680803,7090.309462
min,-24.000000,0.000000,0.000000
25%,1.000000,1.450000,0.000000
50%,3.000000,2.510000,14729.000000
75%,8.000000,4.210000,16456.000000
max,600.000000,569.770000,18144.000000


In [ ]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [ ]:
df.shape

(2000, 8)

  Revenue EDA

In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [ ]:
df["Revenue"].describe()

,Revenue
count,2000.000000
mean,18.907085
std,57.210468
min,-41.400000
25%,3.320000
50%,8.470000
75%,17.700000
max,1627.200000


Monthly Revenue

In [ ]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)


np.int64(2000)

In [ ]:
df["InvoiceDate"].isna().sum()

np.int64(2000)

In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

monthly_revenue = (
    df.groupby(df["InvoiceDate"].dt.to_period("M"))["Revenue"]
      .sum()
      .reset_index()
)

monthly_revenue

,InvoiceDate,Revenue


 outlier checking
 -- Revenue outlier check
 -- Quantity
 -- Treatment


In [ ]:
Q1 = df['Revenue'].quantile(0.25)
Q3 = df['Revenue'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[
    (df['Revenue'] < lower_limit) |
    (df['Revenue'] > upper_limit)
]

print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)
print("Outliers:", len(outliers))

Lower Limit: -18.250000000000004
Upper Limit: 39.27000000000001
Outliers: 175


In [ ]:
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

quantity_outliers = df[
    (df['Quantity'] < lower_limit) |
    (df['Quantity'] > upper_limit)
]

print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)
print("Outliers:", len(quantity_outliers))

Lower Limit: -9.5
Upper Limit: 18.5
Outliers: 232


In [ ]:
print("Revenue outliers:", len(outliers))
print("Quantity outliers:", len(quantity_outliers))

outliers[['Quantity', 'Revenue']].describe()

Revenue outliers: 175
Quantity outliers: 232


,Quantity,Revenue
count,175.000000,175.000000
mean,49.914286,109.833486
std,79.530220,166.267423
min,-24.000000,-41.400000
25%,12.000000,47.000000
50%,30.000000,68.000000
75%,48.000000,101.880000
max,600.000000,1627.200000


Customer Behaviour Analysis
-- customer-wise Orders
-- Revenue
-- order

In [ ]:
customer_analysis = df.groupby('CustomerID').agg(
    total_orders=('InvoiceNo', 'nunique'),
    total_revenue=('Revenue', 'sum'),
    total_items=('Quantity', 'sum')
).reset_index()

customer_analysis.head()

,CustomerID,total_orders,total_revenue,total_items
0,0,7,5521.14,1269
1,12431,1,358.25,107
2,12433,1,1919.14,1852
3,12472,1,-122.30,-40
4,12583,1,855.86,449


In [ ]:
top_customers = customer_analysis.sort_values(
    'total_revenue', ascending=False
).head(10)

top_customers

,CustomerID,total_orders,total_revenue,total_items
0,0,7,5521.14,1269
44,16029,2,3702.12,1676
46,16210,1,2474.74,1070
2,12433,1,1919.14,1852
57,17511,1,1825.74,1568
62,17850,10,1499.34,474
12,13408,1,1024.68,544
37,15485,1,950.09,416
4,12583,1,855.86,449
14,13694,1,842.12,1004


SQL Result Validation
-- Total Revenue
-- Total Customers

In [ ]:
python_total_revenue = df['Revenue'].sum()

print("Python Total Revenue:", python_total_revenue)

Python Total Revenue: 37814.17


In [ ]:
python_customers = df['CustomerID'].nunique()

print("Python Total Customers:", python_customers)

Python Total Customers: 73


RFM Analysis
-- Customer-level RFM table
-- RFM scores
-- Final RFM Score


In [ ]:
import pandas as pd

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (df['InvoiceDate'].max() - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,0,NaN,7,5521.14
1,12431,NaN,1,358.25
2,12433,NaN,1,1919.14
3,12472,NaN,1,-122.30
4,12583,NaN,1,855.86


In [ ]:
rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,F_Score,M_Score
0,0,NaN,7,5521.14,5,5
1,12431,NaN,1,358.25,1,4
2,12433,NaN,1,1919.14,1,5
3,12472,NaN,1,-122.30,1,1
4,12583,NaN,1,855.86,1,5


In [ ]:
rfm['RFM_Score'] = (
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,F_Score,M_Score,RFM_Score
0,0,NaN,7,5521.14,5,5,55
1,12431,NaN,1,358.25,1,4,14
2,12433,NaN,1,1919.14,1,5,15
3,12472,NaN,1,-122.30,1,1,11
4,12583,NaN,1,855.86,1,5,15


In [ ]:
df.to_csv("Online_Retail_Cleaned.csv", index=False)

In [ ]:
from google.colab import files

files.download("Online_Retail_Cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>